# Embeddings experiment: RuBERT + Logistic Regression

В этом ноутбуке проверяется подход к классификации новостей на основе русскоязычных sentence embeddings. Для кодирования текстов используется модель `DeepPavlov/rubert-base-cased-sentence`, а классификатор обучается поверх готовых embedding-векторов.

Цель эксперимента — проверить, улучшит ли русскоязычный RuBERT качество по сравнению с multilingual MiniLM и сможет ли embedding-подход приблизиться к TF-IDF baseline.

In [ ]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer
from pathlib import Path

In [2]:
DATA_PATH = Path("../news_data/cleaned_news_for_model.parquet")

df = pd.read_parquet(DATA_PATH)

df.shape, df.head()

((146619, 10),
      source                                                url archive_date  \
 0  lenta.ru             https://lenta.ru/news/2025/01/01/auto/   2025-01-01   
 1  lenta.ru  https://lenta.ru/news/2025/01/01/v-rossii-stal...   2025-01-01   
 2  lenta.ru  https://lenta.ru/news/2025/01/01/v-rossii-s-1-...   2025-01-01   
 3  lenta.ru  https://lenta.ru/news/2025/01/01/vsu-v-pervye-...   2025-01-01   
 4  lenta.ru  https://lenta.ru/news/2025/01/01/premier-mishu...   2025-01-01   
 
          published_at                                              title  \
 0 2025-01-01 00:00:00  Рост акцизов на топливо, увеличение утильсбора...   
 1 2025-01-01 00:02:00                   В России стало дороже развестись   
 2 2025-01-01 01:23:00  В России с 1 января повысили штрафы за нарушен...   
 3 2025-01-01 01:43:00  ВСУ в первые минуты нового года обстреляли рос...   
 4 2025-01-01 01:57:00   Премьер Мишустин поздравил россиян с Новым годом   
 
                                       

### Промежуточный вывод

Очищенный датасет успешно загружен. Он содержит 146 619 новостей и необходимые поля для моделирования: `title`, `text` и `category_raw`.

В этом эксперименте используется полный текст новости (`text`). Заголовок отдельно не добавляется, чтобы результат был сопоставим с предыдущими экспериментами на TF-IDF и MiniLM.

In [3]:
df_model = df[["title", "text", "category_raw"]].dropna().copy()

df_model["bert_text"] =  df_model["text"].astype(str) #.str[:1500]


df_model = df_model[df_model["bert_text"].str.len() > 0]

df_model.shape

(146619, 4)

### Промежуточный вывод

Для построения embeddings используется полный текст новости. Модель RuBERT применяется как готовый encoder: ее веса не дообучаются на задаче классификации рубрик.

Таким образом, эксперимент проверяет качество универсальных русскоязычных sentence embeddings без fine-tuning.

In [4]:
RUN_ON_SAMPLE = False
SAMPLE_SIZE = 20000

if RUN_ON_SAMPLE:
    df_exp, _ = train_test_split(
        df_model,
        train_size=SAMPLE_SIZE,
        random_state=42,
        stratify=df_model["category_raw"]
    )
else:
    df_exp = df_model.copy()

df_exp["category_raw"].value_counts()

category_raw
Мир                45383
Россия             41352
Экономика          32112
Наука и техника    10168
Спорт               9869
Культура            7735
Name: count, dtype: int64

### Промежуточный вывод

Эксперимент выполняется на полном корпусе из 146 619 новостей. Это делает сравнение с TF-IDF и MiniLM более корректным, но существенно увеличивает время расчета embeddings.

Для быстрых проверок в коде оставлена возможность запуска на стратифицированной подвыборке.

In [5]:
X = df_exp["bert_text"]
y = df_exp["category_raw"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((117295,), (29324,))

### Промежуточный вывод

Данные разделены на train и test в пропорции 80/20 с `random_state=42` и `stratify=y`. Это сохраняет распределение рубрик в обеих выборках.

Размер train-выборки — 117 295 новостей, test-выборки — 29 324 новости.

## Построение RuBERT embeddings

Используется модель `DeepPavlov/rubert-base-cased-sentence`. В отличие от MiniLM, это русскоязычная модель, поэтому ожидается, что она может лучше представлять новости на русском языке.

На этом этапе модель используется только для получения embedding-векторов. Дообучение RuBERT под задачу классификации не выполняется.

In [6]:
MODEL_NAME = "DeepPavlov/rubert-base-cased-sentence"


embedder = SentenceTransformer(MODEL_NAME)

Loading weights: 100%|████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 19391.01it/s]


In [7]:
X_train_emb = embedder.encode(
    X_train.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

X_test_emb = embedder.encode(
    X_test.tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

X_train_emb.shape, X_test_emb.shape

Batches: 100%|█████████████████████████████████████████████████████████████████████| 459/459 [3:09:10<00:00, 24.73s/it]


((117295, 768), (29324, 768))

### Промежуточный вывод

Для каждого текста построен embedding размерности 768. Итоговые матрицы признаков имеют размер `(117295, 768)` для train и `(29324, 768)` для test.

RuBERT embeddings имеют в два раза большую размерность, чем MiniLM embeddings, и заметно дороже вычислительно. Расчет embeddings на полном корпусе занял более 15 часов суммарно для train и test. Это важное практическое ограничение embedding-подхода.

In [8]:
import os

os.makedirs("../news_data/interim", exist_ok=True)

np.save("../news_data/interim/Exp_04_X_train_rubert_base_cased_sentence.npy", X_train_emb)
np.save("../news_data/interim/Exp_04_X_test_rubert_base_cased_sentence.npy", X_test_emb)

y_train.to_csv("../news_data/interim/Exp_04_y_rubert_base_cased_sentence.csv", index=False)
y_test.to_csv("../news_data/interim/Exp_04_y_test_rubert_base_cased_sentence.csv", index=False)

### Промежуточный вывод

RuBERT embeddings сохранены в отдельные `.npy`-файлы. Это позволяет не пересчитывать embeddings при повторном запуске классификатора или дополнительном анализе ошибок.

## Обучение классификатора поверх RuBERT embeddings

После получения dense embeddings обучается Logistic Regression. В этом подходе RuBERT отвечает только за преобразование текста в вектор, а финальное решение о рубрике принимает классическая ML-модель.

Параметр `class_weight="balanced"` используется для учета дисбаланса классов.

In [9]:
clf = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

clf.fit(X_train_emb, y_train)

y_pred = clf.predict(X_test_emb)

In [10]:
accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

Accuracy: 0.8941822398035739
Macro F1: 0.9030353824885924
Weighted F1: 0.893642178412703


### Промежуточный вывод

RuBERT + Logistic Regression показал качество выше, чем MiniLM + Logistic Regression: Accuracy ≈ 0.894, Macro F1 ≈ 0.903, Weighted F1 ≈ 0.894.

Это подтверждает, что русскоязычные sentence embeddings лучше подходят для корпуса новостей на русском языке, чем multilingual MiniLM. Однако результат всё равно заметно ниже TF-IDF + LinearSVC, где Accuracy и Weighted F1 составляют около 0.948.

In [11]:
print(classification_report(y_test, y_pred))

                 precision    recall  f1-score   support

       Культура       0.84      0.97      0.90      1547
            Мир       0.91      0.92      0.91      9077
Наука и техника       0.83      0.96      0.89      2034
         Россия       0.91      0.82      0.86      8270
          Спорт       0.95      0.99      0.97      1974
      Экономика       0.87      0.89      0.88      6422

       accuracy                           0.89     29324
      macro avg       0.89      0.92      0.90     29324
   weighted avg       0.90      0.89      0.89     29324



### Промежуточный вывод

RuBERT показывает лучшее качество почти по всем классам по сравнению с MiniLM. Особенно хорошо классифицируются `Спорт`, `Мир`, `Культура` и `Наука и техника`.

Наиболее сложной остается рубрика `Россия`: F1-score около 0.86. Это связано с тем, что новости о России часто пересекаются с экономической, международной и общественно-политической повесткой.

In [12]:
result = pd.DataFrame([{
    "experiment": "04_rubert_base_cased_sentence_text_logreg",
    "model": MODEL_NAME,
    "input": "text",
    "classifier": "LogisticRegression",
    "sample_size": len(df_exp),
    "accuracy": accuracy,
    "macro_f1": macro_f1,
    "weighted_f1": weighted_f1
}])

os.makedirs("../reports", exist_ok=True)

report_path = "../reports/rubert_base_cased_sentence_results.csv"

result.to_csv(
    report_path,
    mode="a",
    header=not os.path.exists(report_path),
    index=False
)

result

,experiment,model,input,classifier,sample_size,accuracy,macro_f1,weighted_f1
0,04_rubert_base_cased_sentence_text_logreg,DeepPavlov/rubert-base-cased-sentence,text,LogisticRegression,146619,0.894182,0.903035,0.893642


### Промежуточный вывод

Результаты RuBERT-эксперимента сохранены в CSV-файл. Их можно использовать в для сравнения моделей вместе с TF-IDF, MiniLM и LLM-подходами.

## Анализ ошибок

Для анализа ошибок строится confusion matrix. Она позволяет понять, какие рубрики RuBERT-модель путает чаще всего и совпадает ли структура ошибок с предыдущими подходами.

In [13]:
labels = sorted(y_test.unique())

cm = confusion_matrix(y_test, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels]
)

cm_df

,pred_Культура,pred_Мир,pred_Наука и техника,pred_Россия,pred_Спорт,pred_Экономика
true_Культура,1501,7,3,20,5,11
true_Мир,46,8324,135,321,20,231
true_Наука и техника,2,32,1944,27,1,28
true_Россия,149,615,125,6771,52,558
true_Спорт,4,4,2,10,1950,4
true_Экономика,87,196,129,264,15,5731


### Промежуточный вывод

Как и в предыдущих экспериментах, основные ошибки сосредоточены между рубриками `Россия`, `Мир` и `Экономика`. Это подтверждает, что именно эти классы являются наиболее близкими по содержанию.

По сравнению с MiniLM абсолютное число ошибок меньше, но по сравнению с TF-IDF + LinearSVC ошибки между крупными классами всё еще заметно выше.

In [14]:
cm_norm = cm / cm.sum(axis=1, keepdims=True)

cm_norm_df = pd.DataFrame(
    cm_norm,
    index=[f"true_{label}" for label in labels],
    columns=[f"pred_{label}" for label in labels]
)

cm_norm_df.round(3)

,pred_Культура,pred_Мир,pred_Наука и техника,pred_Россия,pred_Спорт,pred_Экономика
true_Культура,0.970,0.005,0.002,0.013,0.003,0.007
true_Мир,0.005,0.917,0.015,0.035,0.002,0.025
true_Наука и техника,0.001,0.016,0.956,0.013,0.000,0.014
true_Россия,0.018,0.074,0.015,0.819,0.006,0.067
true_Спорт,0.002,0.002,0.001,0.005,0.988,0.002
true_Экономика,0.014,0.031,0.020,0.041,0.002,0.892


### Промежуточный вывод

Нормализованная confusion matrix показывает, что главная проблема RuBERT-модели — классификация рубрики `Россия`: около 7.4% таких новостей предсказываются как `Мир`, еще около 6.7% — как `Экономика`.

Это лучше, чем у MiniLM по части классов, но всё еще слабее TF-IDF + LinearSVC. Вероятно, универсальные embeddings хуже фиксируют редакционные границы рубрик, чем TF-IDF-признаки, напрямую связанные с тематической лексикой.

In [15]:
errors = pd.DataFrame({
    "text": X_test,
    "true": y_test,
    "pred": y_pred
})

errors = errors[errors["true"] != errors["pred"]]

errors.head(20)

,text,true,pred
42845,Правительство одобрило увеличение пособия по б...,Россия,Экономика
83817,СК начал проверку причин смерти экс-чиновника ...,Россия,Культура
79924,ДОМ.РФ профинансировал реконструкцию троллейбу...,Россия,Экономика
103694,«Лента» запустила рандомайзер карт с советами ...,Экономика,Россия
41823,Мединский: Россия будет ждать украинскую делег...,Россия,Мир
103969,Reuters: Число жертв крупного пожара в Гонконг...,Мир,Экономика
90761,Песков: Россия сохраняет открытость к мирному ...,Россия,Мир
128187,Посол РФ во Франции Мешков: Получено 170 заяво...,Мир,Россия
138793,Сенатор Карасин: Наземная операция США в Иране...,Россия,Мир
128332,Дмитриев посчитал фото брата Карла III в Лувре...,Мир,Культура


### Промежуточный вывод

Просмотр отдельных ошибок показывает, что многие новости находятся на границе нескольких рубрик. Например, тексты о российских официальных лицах, международных переговорах, санкциях, экономических мерах и государственных решениях могут быть отнесены сразу к нескольким тематическим областям.

Это объясняет, почему основные ошибки повторяются у разных моделей: проблема связана не только с конкретным алгоритмом, но и с близостью самих рубрик.

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "TF-IDF + Logistic Regression",
        "accuracy": 0.9408,
        "macro_f1": 0.9490,
        "weighted_f1": 0.9407,
    },
    {
        "model": "TF-IDF + LinearSVC",
        "accuracy": 0.9482,
        "macro_f1": 0.9570,
        "weighted_f1": 0.9482,
    },
    {
        "model": "MiniLM + Logistic Regression",
        "accuracy": 0.8680,
        "macro_f1": 0.8740,
        "weighted_f1": 0.8680,
    },
    {
        "model": "RuBERT + Logistic Regression",
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    },
])

comparison

### Промежуточный вывод

RuBERT заметно улучшил результат относительно MiniLM, но не приблизился к TF-IDF + LinearSVC. Разрыв с лучшей TF-IDF-моделью составляет около 5.4 п.п. по Accuracy и Weighted F1.

## Ключевой вывод

RuBERT embeddings + Logistic Regression показали качество выше, чем MiniLM embeddings: Accuracy ≈ 0.894, Macro F1 ≈ 0.903, Weighted F1 ≈ 0.894. Это подтверждает преимущество русскоязычной embedding-модели для корпуса новостей на русском языке.

Однако RuBERT без fine-tuning всё равно уступает TF-IDF + LinearSVC. Для данной задачи тематическая лексика оказывается более сильным сигналом, чем универсальные dense embeddings без дообучения.